In [1]:
"""
musica_dust_ginoux_box_model.py
===============================
MUSICA box-model for the Ginoux et al. (2001) / GOCART-2G dust
emission scheme.

The physics lives in dust_ginoux.py (untouched science).

This file builds the MICM mechanism, creates the solver, and time-steps. 
Each step it asks the science module for F_p, 
divides by dz to get a volumetric emission rate, 
and sets it on the MICM
state via the EMIS.* user-defined parameter convention.

LOSS:  state.set_user_defined_rate_parameters({"LOSS.Hg0_drydep": [k]})
EMIS:  state.set_user_defined_rate_parameters({"EMIS.dust_emis_bin_i": [e_i]})

The static and synthetic-dynamic inputs below mirror dust_ginoux.py, 
so this file is the MICM analog of that demo: same numbers,
same threshold computation, but solved by the Rosenbrock integrator instead
of an explicit Forward Euler step.
"""

import numpy as np
import musica
import musica.mechanism_configuration as mc

from dust_ginoux import (
    dust_emission,
    threshold_velocity,
    RADIUS_DEFAULT,
    RHOP_DEFAULT,
    SP_DEFAULT,
    C_DEFAULT,
    LAND,
)

n_bins = len(SP_DEFAULT)
print(n_bins)

5


In [29]:
import numpy as np
import musica
import musica.mechanism_configuration as mc

from dust_ginoux import (
    dust_emission,
    threshold_velocity,
    RADIUS_DEFAULT,
    RHOP_DEFAULT,
    SP_DEFAULT,
    C_DEFAULT,
    LAND,
)

n_bins = len(SP_DEFAULT)
print(n_bins)

5


In [34]:

DUST_BIN_MW = 1.0  # kg/mol, dimensional placeholder (see note above)

dust_species = [
    mc.Species(name=f"dust_bin_{i+1}", molecular_weight_kg_mol=DUST_BIN_MW)
    for i in range(n_bins)
]
gas = mc.Phase(name="gas", species=dust_species)

dust_emissions = [
    mc.Emission(
        name=f"dust_emis_bin_{i+1}",
        scaling_factor=1.0,
        products=[dust_species[i]],
        gas_phase=gas,
    )
    for i in range(n_bins)
]

mechanism = mc.Mechanism(
    name="quacs_dust_ginoux",
    species=dust_species,
    phases=[gas],
    reactions=dust_emissions,
)

In [36]:
S      = np.array([[0.5]])    # erodibility [-]
oro    = np.array([[LAND]])   # land mask
frlake = np.array([[0.0]])    # no lake
rhoa   = np.array([[1.2]])    # near-surface air density [kg/m^3]

# Dry-soil threshold per bin via Marticorena & Bergametti (1995),
# then bin-mean to feed dust_emission (which takes a single u_t [x, y]).
u_t_bins = threshold_velocity(RADIUS_DEFAULT, RHOP_DEFAULT, rhoa)
u_t      = np.array([[float(u_t_bins[0, 0, :].mean())]])

# ---------------------------------------------------------------------------
# 4. Helper: surface flux -> MICM emission rate
#    F_p [kg m-2 s-1] / dz [m] = e [kg m-3 s-1]   (the source for dC/dt = +e)
#    EMIS analog of K20's k = Vd * 0.01 / H for LOSS.
# ---------------------------------------------------------------------------
def compute_emission_rates(w10m, gwet, dz):
    F_p = dust_emission(
        w10m=w10m, u_t=u_t, gwet=gwet, oro=oro,
        frlake=frlake, S=S, s_p=SP_DEFAULT, C=C_DEFAULT,
    )  # [x, y, n_bins]
    return {
        f"EMIS.dust_emis_bin_{i+1}": [F_p[0, 0, i] / dz]
        for i in range(n_bins)
    }

In [51]:
solver = musica.MICM(
    mechanism=mechanism,
    solver_type=musica.SolverType.rosenbrock_standard_order,
)

state = solver.create_state(number_of_grid_cells=1)

state.set_conditions([300.0], [101325.0])  # T [K], p [Pa]

state.set_concentrations({sp.name: [0.0] for sp in dust_species})


In [52]:

dt      = 120.0               # s, model time step
dz      = 50.0                # m, lowest-layer thickness (from MPAS-A)
n_steps = 30                  # 12 * 300 s = 1 hour

w10m    = np.array([[12.0]])  # m/s, synthetic (constant)

gwet    = np.array([[ 0.1]])  # [-], synthetic (constant)

print("--- MUSICA dust-emission box model ---")
print(f"  w10m={w10m[0,0]:.1f} m/s, gwet={gwet[0,0]:.2f}, "
      f"u_t (bin-mean)={u_t[0,0]:.3f} m/s")
print(f"  dz={dz:.1f} m, dt={dt:.0f} s, n_steps={n_steps}, C={C_DEFAULT}\n")

rates = compute_emission_rates(w10m, gwet, dz)
state.set_user_defined_rate_parameters(rates)

e_per_bin = np.array([rates[f"EMIS.dust_emis_bin_{i+1}"][0]
                      for i in range(n_bins)])     # [kg m-3 s-1]
times     = np.zeros(n_steps + 1)
concs     = np.zeros((n_steps + 1, n_bins))         # [kg m-3]

for step in range(n_steps):
    solver.solve(state, dt)
    
    state_conc = state.get_concentrations()

    
    for i in range(n_bins):
        concs[step + 1, i] = state_conc[f"dust_bin_{i+1}"][0]
    times[step + 1] = (step + 1) * dt

--- MUSICA dust-emission box model ---
  w10m=12.0 m/s, gwet=0.10, u_t (bin-mean)=1.224 m/s
  dz=50.0 m, dt=120 s, n_steps=30, C=0.088



In [54]:
# ---- print evolution + closed-form check at every step --------------
print(f"{'time (min)':>10}  {'total dust [kg/m3]':>22}  "
      f"{'expected [kg/m3]':>22}  match")

print("-" * 70)

all_pass = True

for step in range(n_steps + 1):
    
    total_actual   = concs[step].sum()

    total_expected = e_per_bin.sum() * times[step]
    
    ok = np.isclose(total_actual, total_expected, rtol=1e-10)
    
    all_pass = all_pass and ok
    
    print(f"{times[step]/60:>10.2f}  {total_actual:>22.6e}  "
          f"{total_expected:>22.6e}  {'ok' if ok else 'FAIL'}")


time (min)      total dust [kg/m3]        expected [kg/m3]  match
----------------------------------------------------------------------
      0.00            0.000000e+00            0.000000e+00  ok
      2.00            1.802468e+02            1.802468e+02  ok
      4.00            3.604936e+02            3.604936e+02  ok
      6.00            5.407405e+02            5.407405e+02  ok
      8.00            7.209873e+02            7.209873e+02  ok
     10.00            9.012341e+02            9.012341e+02  ok
     12.00            1.081481e+03            1.081481e+03  ok
     14.00            1.261728e+03            1.261728e+03  ok
     16.00            1.441975e+03            1.441975e+03  ok
     18.00            1.622221e+03            1.622221e+03  ok
     20.00            1.802468e+03            1.802468e+03  ok
     22.00            1.982715e+03            1.982715e+03  ok
     24.00            2.162962e+03            2.162962e+03  ok
     26.00            2.343209e+03          